# RAD Framework — RelBench H&M User-Churn

Applies the RAD_fixed framework to the `rel-hm` `user-churn` task:
- **At-risk filtering**: Identify users with warning signs (low activity/inactivity) and DROP them; remaining churners = true anomalies
- **Rule mining**: RF → LLM SQL refinement → DuckDB evaluation on customer activity features
- **Rule augmentation**: binary rule-violation indicators injected into the RAD autoencoder
- **RAD model**: feature MSE + structure BCE + pairwise rank loss on combined feat+struct score

In [ ]:
tail -f results/hm/gae_norules_10seed/run.log

In [10]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU required but CUDA not available.")

torch.cuda.set_device(0)
device = torch.device("cuda")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: NVIDIA RTX 6000 Ada Generation


In [11]:
import os

rdl_graph_cache_path = os.path.join("data", "relbench_rdl_graph_cache_hm.pt")
rdl_stats_cache_path = os.path.join("data", "relbench_rdl_col_stats_cache_hm.pt")

if os.path.exists(rdl_graph_cache_path) and os.path.exists(rdl_stats_cache_path):
    rdl_data = torch.load(rdl_graph_cache_path, weights_only=False)
    rdl_col_stats_dict = torch.load(rdl_stats_cache_path, weights_only=False)
    print({"loaded_graph_cache": rdl_graph_cache_path, "loaded_stats_cache": rdl_stats_cache_path})
else:
    rdl_data = None
    rdl_col_stats_dict = None
    print("No cache found — will build graph from scratch in setup cell.")

No cache found — will build graph from scratch in setup cell.


In [15]:
import copy

import gc

import math

import os

from typing import Any, Dict, List, Optional



import numpy as np

import pandas as pd

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

from torch.nn import Embedding, ModuleDict

from torch_geometric.data import HeteroData

from torch_geometric.nn import MLP

from torch_geometric.seed import seed_everything

from torch_frame.data.stats import StatType

from torch_frame.config.text_embedder import TextEmbedderConfig

from sentence_transformers import SentenceTransformer

from sklearn.metrics import average_precision_score, roc_auc_score

from tqdm import tqdm



from relbench.datasets import get_dataset

from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph

from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder

from relbench.modeling.utils import get_stype_proposal

from relbench.tasks import get_task



seed_everything(42)



rdl_dataset = get_dataset("rel-hm", download=True)

rdl_task = get_task("rel-hm", "user-churn", download=True)

rdl_db = rdl_dataset.get_db()



rdl_train_table = rdl_task.get_table("train")

rdl_val_table = rdl_task.get_table("val")

rdl_test_table = rdl_task.get_table("test")



_rdl_orig_val_table = copy.deepcopy(rdl_val_table)

_rdl_orig_train_table = copy.deepcopy(rdl_train_table)



# Build customer activity events from H&M transactions.

txn_events = rdl_db.table_dict["transactions"].df[["customer_id", "t_dat"]].dropna().copy()

txn_events["customer_id"] = txn_events["customer_id"].astype(np.int64)

txn_events["t_dat"] = pd.to_datetime(txn_events["t_dat"])

txn_events = txn_events.sort_values(["customer_id", "t_dat"]).reset_index(drop=True)

txn_events["cum_txns"] = txn_events.groupby("customer_id").cumcount() + 1





def _activity_features_for_targets(target_df: pd.DataFrame, lookback_days: int = 30) -> pd.DataFrame:

    q = target_df[["customer_id", "timestamp"]].copy()

    q["customer_id"] = q["customer_id"].astype(np.int64)

    q["timestamp"] = pd.to_datetime(q["timestamp"])

    q["_row_id"] = np.arange(len(q), dtype=np.int64)

    q = q.sort_values(["timestamp", "customer_id", "_row_id"]).reset_index(drop=True)

    right = txn_events[["customer_id", "t_dat", "cum_txns"]]

    now_join = pd.merge_asof(

        q[["customer_id", "timestamp", "_row_id"]].sort_values(["timestamp", "customer_id"]),

        right.sort_values(["t_dat", "customer_id"]),

        left_on="timestamp", right_on="t_dat",

        by="customer_id", direction="backward", allow_exact_matches=False,

    )

    q["txns_before_now"] = now_join["cum_txns"].fillna(0).astype(np.int64)

    q["last_t_dat"] = now_join["t_dat"]

    q["window_start"] = q["timestamp"] - pd.Timedelta(days=lookback_days)

    start_join = pd.merge_asof(

        q[["customer_id", "window_start", "_row_id"]].sort_values(["window_start", "customer_id"]),

        right.sort_values(["t_dat", "customer_id"]),

        left_on="window_start", right_on="t_dat",

        by="customer_id", direction="backward", allow_exact_matches=False,

    )

    q["txns_before_window"] = start_join["cum_txns"].fillna(0).astype(np.int64)

    q["recent_txns"] = q["txns_before_now"] - q["txns_before_window"]

    delta = q["timestamp"] - q["last_t_dat"]

    q["days_since_last_txn"] = delta.dt.days.fillna(9999).astype(np.int64)

    q = q.sort_values("_row_id").reset_index(drop=True)

    return q[["recent_txns", "days_since_last_txn"]]





def _heuristic_unexpected_labels(

    target_df, activity_df, thresholds=None,

    recent_q=0.75, recency_q=0.35, min_recent_floor=1, max_days_cap=60,

):

    raw = target_df["churn"].astype(np.int64).to_numpy()

    if thresholds is None:

        non_churn = raw == 0

        if int(non_churn.sum()) == 0:

            recent_min, max_days_since = min_recent_floor, max_days_cap

        else:

            recent_min = max(min_recent_floor, int(np.ceil(np.quantile(

                activity_df.loc[non_churn, "recent_txns"].to_numpy(), recent_q))))

            max_days_since = min(max_days_cap, max(0, int(np.floor(np.quantile(

                activity_df.loc[non_churn, "days_since_last_txn"].to_numpy(), recency_q)))))

        thresholds = {"recent_min": recent_min, "max_days_since": max_days_since}

    unexpected = (

        (raw == 1)

        & (activity_df["recent_txns"].to_numpy() >= thresholds["recent_min"])

        & (activity_df["days_since_last_txn"].to_numpy() <= thresholds["max_days_since"])

    )

    keep_mask = (raw == 0) | unexpected

    return unexpected.astype(np.int64), keep_mask, thresholds





rdl_activity_train = _activity_features_for_targets(rdl_train_table.df, lookback_days=30)

rdl_activity_val = _activity_features_for_targets(rdl_val_table.df, lookback_days=30)



rdl_train_unexpected, rdl_train_keep, rdl_heuristic_thresholds = _heuristic_unexpected_labels(

    rdl_train_table.df, rdl_activity_train)

rdl_val_unexpected, rdl_val_keep, _ = _heuristic_unexpected_labels(

    rdl_val_table.df, rdl_activity_val, thresholds=rdl_heuristic_thresholds)



rdl_train_table.df = rdl_train_table.df.loc[rdl_train_keep].copy()

rdl_train_table.df["churn"] = rdl_train_unexpected[rdl_train_keep]

rdl_val_table.df = rdl_val_table.df.loc[rdl_val_keep].copy()

rdl_val_table.df["churn"] = rdl_val_unexpected[rdl_val_keep]



# Match val positive prevalence to train by downsampling val positives.

_rdl_train_pos_rate = float(rdl_train_table.df["churn"].mean())

_rdl_val_pos_df = rdl_val_table.df[rdl_val_table.df["churn"] == 1]

_rdl_val_neg_df = rdl_val_table.df[rdl_val_table.df["churn"] == 0]

if 0.0 < _rdl_train_pos_rate < 1.0 and len(_rdl_val_pos_df) > 0 and len(_rdl_val_neg_df) > 0:

    _n_pos_target = int(round((_rdl_train_pos_rate / (1.0 - _rdl_train_pos_rate)) * len(_rdl_val_neg_df)))

    _n_pos_target = min(_n_pos_target, len(_rdl_val_pos_df))

    if _n_pos_target < len(_rdl_val_pos_df):

        _rng_eq = np.random.default_rng(42)

        _keep_pos_idx = _rng_eq.choice(_rdl_val_pos_df.index.to_numpy(), size=_n_pos_target, replace=False)

        _keep_idx = np.sort(np.concatenate([_rdl_val_neg_df.index.to_numpy(), _keep_pos_idx]))

        rdl_val_table.df = rdl_val_table.df.loc[_keep_idx].copy()



print("Heuristic thresholds:", rdl_heuristic_thresholds)

print("Train:", {"kept": int(rdl_train_keep.sum()), "unexpected": int(rdl_train_unexpected.sum())})

print("Val:  ", {"kept": int(rdl_val_keep.sum()), "unexpected": int(rdl_val_unexpected.sum())})

print(f"Train unexpected rate: {rdl_train_table.df['churn'].mean():.4f}")

print(f"Val unexpected rate:   {rdl_val_table.df['churn'].mean():.4f}")



rdl_tune_metric = "roc_auc"

rdl_entity_table = rdl_task.entity_table  # "customer"



# Build or load H&M graph cache.

class GloveTextEmbedding:

    def __init__(self, device="cpu", encode_batch_size=64):

        self.encode_batch_size = int(encode_batch_size)

        self.model = SentenceTransformer(

            "sentence-transformers/average_word_embeddings_glove.6B.300d", device=device)



    def __call__(self, sentences):

        return torch.from_numpy(self.model.encode(

            sentences, batch_size=self.encode_batch_size, show_progress_bar=False, convert_to_numpy=True))



rdl_col_to_stype = get_stype_proposal(rdl_db)

rdl_text_embedder_cfg = TextEmbedderConfig(

    text_embedder=GloveTextEmbedding(device="cpu", encode_batch_size=64), batch_size=64)



if rdl_data is not None and rdl_col_stats_dict is not None:

    print("Using preloaded cached graph artifacts.")

else:

    rdl_data, rdl_col_stats_dict = make_pkey_fkey_graph(

        rdl_db,

        col_to_stype_dict=rdl_col_to_stype,

        text_embedder_cfg=rdl_text_embedder_cfg,

        cache_dir=None,

    )

    torch.save(rdl_data, rdl_graph_cache_path)

    torch.save(rdl_col_stats_dict, rdl_stats_cache_path)



gc.collect()

torch.cuda.empty_cache()

print(rdl_data)



from torch_geometric.loader import NeighborLoader

rdl_loader_dict = {}

for split, table in [("train", rdl_train_table), ("val", rdl_val_table), ("test", rdl_test_table)]:

    table_input = get_node_train_table_input(table=table, task=rdl_task)

    rdl_loader_dict[split] = NeighborLoader(

        rdl_data,

        num_neighbors=[32, 32],

        time_attr="time",

        input_nodes=table_input.nodes,

        input_time=table_input.time,

        transform=table_input.transform,

        batch_size=1024,

        temporal_strategy="uniform",

        shuffle=(split == "train"),

        num_workers=0,

        persistent_workers=False,

        pin_memory=True,

    )

print("Loaders:", {s: len(l) for s, l in rdl_loader_dict.items()})


Loading Database object from <RELBENCH_CACHE>/rel-hm/db...
Done in 0.83 seconds.
Heuristic thresholds: {'recent_min': 10, 'max_days_since': 3}
Train: {'kept': 932750, 'unexpected': 238007}
Val:   {'kept': 18375, 'unexpected': 4044}
Train unexpected rate: 0.2552
Val unexpected rate:   0.2201


Embedding raw data in mini-batch: 100%|██████████| 1650/1650 [00:03<00:00, 449.69it/s]


HeteroData(
  transactions={
    tf=TensorFrame([15187287, 3]),
    time=[15187287],
  },
  customer={ tf=TensorFrame([1371980, 6]) },
  article={ tf=TensorFrame([105542, 24]) },
  (transactions, f2p_customer_id, customer)={ edge_index=[2, 15187287] },
  (customer, rev_f2p_customer_id, transactions)={ edge_index=[2, 15187287] },
  (transactions, f2p_article_id, article)={ edge_index=[2, 15187287] },
  (article, rev_f2p_article_id, transactions)={ edge_index=[2, 15187287] }
)
Loaders: {'train': 911, 'val': 18, 'test': 73}


In [18]:
# ── Customer flat features ────────────────────────────────────────────────────

# Aggregate activity stats per (customer_id, timestamp) target row,

# computed from transaction data strictly before each timestamp.



txn_df  = rdl_db.table_dict["transactions"].df.copy()

txn_df["customer_id"]  = txn_df["customer_id"].astype(np.int64)

txn_df["t_dat"]  = pd.to_datetime(txn_df["t_dat"])

_price_col = "price" if "price" in txn_df.columns else None





def _build_customer_flat(target_df, activity_df, lookback_90=90):

    base = target_df[["customer_id", "timestamp"]].copy()

    base["customer_id"] = base["customer_id"].astype(np.int64)

    base["timestamp"]   = pd.to_datetime(base["timestamp"])

    base = base.reset_index(drop=True)



    base["recent_txns_30d"]     = activity_df["recent_txns"].values

    base["days_since_last_txn"] = activity_df["days_since_last_txn"].values



    # Total reviews before timestamp.

    _right = txn_events[["customer_id", "t_dat", "cum_txns"]]

    _j = pd.merge_asof(

        base[["customer_id", "timestamp"]].sort_values(["timestamp", "customer_id"]),

        _right.sort_values(["t_dat", "customer_id"]),

        left_on="timestamp", right_on="t_dat",

        by="customer_id", direction="backward", allow_exact_matches=False,

    )

    base["total_txns"] = _j["cum_txns"].fillna(0).astype(np.int64).values



    # 90-day reviews.

    _q2 = base[["customer_id", "timestamp"]].copy()

    _q2["window_90"] = _q2["timestamp"] - pd.Timedelta(days=lookback_90)

    _j2 = pd.merge_asof(

        _q2.sort_values(["window_90", "customer_id"]),

        _right.sort_values(["t_dat", "customer_id"]),

        left_on="window_90", right_on="t_dat",

        by="customer_id", direction="backward", allow_exact_matches=False,

    )

    base["recent_txns_90d"] = np.maximum(

        0, base["total_txns"].values - _j2["cum_txns"].fillna(0).astype(np.int64).values

    )



    # Rating stats (global aggregate — fine for pattern mining).

    if _price_col is not None:

        _rated = txn_df[["customer_id", _price_col]].copy()

        _rated[_price_col] = pd.to_numeric(_rated[_price_col], errors="coerce")

        _rated = _rated.dropna(subset=[_price_col])

        _agg = _rated.groupby("customer_id")[_price_col].agg(

            mean_price="mean", price_std="std", min_price="min", max_price="max"

        ).reset_index()

        base = base.merge(_agg, on="customer_id", how="left")

        for c, v in [("mean_price", 3.0), ("price_std", 0.0), ("min_price", 3.0), ("max_price", 3.0)]:

            base[c] = base[c].fillna(v)

    else:

        for c in ["mean_price", "price_std", "min_price", "max_price"]:

            base[c] = 0.0



    # Distinct products.

    if "article_id" in txn_df.columns:

        _div = txn_df.groupby("customer_id")["article_id"].nunique().rename("distinct_articles")

        base = base.merge(_div.reset_index(), on="customer_id", how="left")

        base["distinct_articles"] = base["distinct_articles"].fillna(0).astype(np.int64)

    else:

        base["distinct_articles"] = 0



    # Review span.

    _span = txn_df.groupby("customer_id")["t_dat"].agg(

        first_review="min", last_review="max").reset_index()

    _span["txn_span_days"] = (

        (_span["last_review"] - _span["first_review"]).dt.days.fillna(0).astype(np.int64))

    base = base.merge(_span[["customer_id", "txn_span_days"]], on="customer_id", how="left")

    base["txn_span_days"] = base["txn_span_days"].fillna(0).astype(np.int64)



    base["low_price_flag"] = (base["mean_price"] < 20.0).astype(np.int8)

    return base





# Recompute activity features on the CURRENT (post-filter, post-downsample) train table

# to ensure row alignment with the filtered/downsampled rdl_train_table.df.

_activity_train_kept = _activity_features_for_targets(rdl_train_table.df, lookback_days=30)

train_flat = _build_customer_flat(rdl_train_table.df, _activity_train_kept)

print(f"Train flat features: {train_flat.shape}")

print(train_flat.dtypes)


Train flat features: (567146, 13)
customer_id                     int64
timestamp              datetime64[ns]
recent_txns_30d                 int64
days_since_last_txn             int64
total_txns                      int64
recent_txns_90d                 int64
mean_price                    float64
price_std                     float64
min_price                     float64
max_price                     float64
distinct_articles               int64
txn_span_days                   int64
low_price_flag                   int8
dtype: object


In [16]:
# ── At-risk user identification and filtering ────────────────────────────────────

# Instead of labeling "unexpected churners," we identify at-risk users (showing warning signs)

# and DROP them entirely from the dataset. Remaining churners are true anomalies.

# Target: 1-2% anomaly rate in final dataset.



# Fallback bootstrap if setup cell was not run successfully.

if "_activity_features_for_targets" not in globals():

    if "rdl_db" not in globals() or "rdl_train_table" not in globals() or "rdl_val_table" not in globals():

        raise RuntimeError(

            "Setup variables are missing. Run Cells 2-4 first (GPU, cache, dataset/task setup)."

        )

    txn_events = rdl_db.table_dict["transactions"].df[["customer_id", "t_dat"]].dropna().copy()

    txn_events["customer_id"] = txn_events["customer_id"].astype(np.int64)

    txn_events["t_dat"] = pd.to_datetime(txn_events["t_dat"])

    txn_events = txn_events.sort_values(["customer_id", "t_dat"]).reset_index(drop=True)

    txn_events["cum_txns"] = txn_events.groupby("customer_id").cumcount() + 1



    def _activity_features_for_targets(target_df: pd.DataFrame, lookback_days: int = 30) -> pd.DataFrame:

        q = target_df[["customer_id", "timestamp"]].copy()

        q["customer_id"] = q["customer_id"].astype(np.int64)

        q["timestamp"] = pd.to_datetime(q["timestamp"])

        q["_row_id"] = np.arange(len(q), dtype=np.int64)

        q = q.sort_values(["timestamp", "customer_id", "_row_id"]).reset_index(drop=True)

        right = txn_events[["customer_id", "t_dat", "cum_txns"]]

        now_join = pd.merge_asof(

            q[["customer_id", "timestamp", "_row_id"]].sort_values(["timestamp", "customer_id"]),

            right.sort_values(["t_dat", "customer_id"]),

            left_on="timestamp", right_on="t_dat",

            by="customer_id", direction="backward", allow_exact_matches=False,

        )

        q["txns_before_now"] = now_join["cum_txns"].fillna(0).astype(np.int64)

        q["last_t_dat"] = now_join["t_dat"]

        q["window_start"] = q["timestamp"] - pd.Timedelta(days=lookback_days)

        start_join = pd.merge_asof(

            q[["customer_id", "window_start", "_row_id"]].sort_values(["window_start", "customer_id"]),

            right.sort_values(["t_dat", "customer_id"]),

            left_on="window_start", right_on="t_dat",

            by="customer_id", direction="backward", allow_exact_matches=False,

        )

        q["txns_before_window"] = start_join["cum_txns"].fillna(0).astype(np.int64)

        q["recent_txns"] = q["txns_before_now"] - q["txns_before_window"]

        delta = q["timestamp"] - q["last_t_dat"]

        q["days_since_last_txn"] = delta.dt.days.fillna(9999).astype(np.int64)

        q = q.sort_values("_row_id").reset_index(drop=True)

        return q[["recent_txns", "days_since_last_txn"]]



def _identify_at_risk_users(

    target_df, activity_df, thresholds=None,

    recent_q=0.10, recency_q=0.90, min_recent_floor=1, max_days_cap=60,

):

    """

    Identify 'at-risk' users: those showing warning signs of churn (low activity, inactivity).

    These users will be DROPPED from the dataset to create a harder anomaly detection task.



    At-risk defined as: low recent activity OR high recency (inactive).

    Aggressive thresholds: drop ~95-98% of users to achieve 1-2% anomaly rate.

    """

    raw = target_df["churn"].astype(np.int64).to_numpy()

    if thresholds is None:

        non_churn = raw == 0

        if int(non_churn.sum()) == 0:

            recent_max, min_days_since = min_recent_floor, max_days_cap

        else:

            # At-risk thresholds: very aggressive to drop most users

            recent_max = max(min_recent_floor, int(np.ceil(np.quantile(

                activity_df.loc[non_churn, "recent_txns"].to_numpy(), recent_q))))

            min_days_since = min(max_days_cap, max(0, int(np.floor(np.quantile(

                activity_df.loc[non_churn, "days_since_last_txn"].to_numpy(), recency_q)))))

        thresholds = {"recent_max": recent_max, "min_days_since": min_days_since}



    at_risk = (

        (activity_df["recent_txns"].to_numpy() <= thresholds["recent_max"])

        | (activity_df["days_since_last_txn"].to_numpy() >= thresholds["min_days_since"])

    )

    keep_mask = ~at_risk

    anomaly_labels = raw * keep_mask.astype(np.int64)

    return anomaly_labels, keep_mask, thresholds



rdl_activity_train = _activity_features_for_targets(rdl_train_table.df, lookback_days=30)

rdl_activity_val   = _activity_features_for_targets(rdl_val_table.df, lookback_days=30)



# Identify at-risk users and get thresholds from train set

rdl_train_anomaly_labels, rdl_train_keep, rdl_at_risk_thresholds = _identify_at_risk_users(

    rdl_train_table.df, rdl_activity_train)

rdl_val_anomaly_labels, rdl_val_keep, _ = _identify_at_risk_users(

    rdl_val_table.df, rdl_activity_val, thresholds=rdl_at_risk_thresholds)



# Filter out at-risk users from both train and val

rdl_train_table.df = rdl_train_table.df.loc[rdl_train_keep].copy()

rdl_train_table.df["churn"] = rdl_train_anomaly_labels[rdl_train_keep]

rdl_val_table.df = rdl_val_table.df.loc[rdl_val_keep].copy()

rdl_val_table.df["churn"] = rdl_val_anomaly_labels[rdl_val_keep]



# Downsample to achieve target 1-2% anomaly rate (use 1.5% as target)

_target_anom_rate = 0.015

_rdl_train_pos_df = rdl_train_table.df[rdl_train_table.df["churn"] == 1]

_rdl_train_neg_df = rdl_train_table.df[rdl_train_table.df["churn"] == 0]

if 0.0 < _target_anom_rate < 1.0 and len(_rdl_train_pos_df) > 0 and len(_rdl_train_neg_df) > 0:

    _n_pos_target_train = int(round((_target_anom_rate / (1.0 - _target_anom_rate)) * len(_rdl_train_neg_df)))

    _n_pos_target_train = min(_n_pos_target_train, len(_rdl_train_pos_df))

    if _n_pos_target_train < len(_rdl_train_pos_df):

        _rng_train = np.random.default_rng(42)

        _keep_pos_idx_train = _rng_train.choice(_rdl_train_pos_df.index.to_numpy(),

                                                size=_n_pos_target_train, replace=False)

        _keep_idx_train = np.sort(np.concatenate([_rdl_train_neg_df.index.to_numpy(), _keep_pos_idx_train]))

        rdl_train_table.df = rdl_train_table.df.loc[_keep_idx_train].copy()



# Match val to train rate via downsampling

_rdl_train_pos_rate = float(rdl_train_table.df["churn"].mean())

_rdl_val_pos_df = rdl_val_table.df[rdl_val_table.df["churn"] == 1]

_rdl_val_neg_df = rdl_val_table.df[rdl_val_table.df["churn"] == 0]

if 0.0 < _rdl_train_pos_rate < 1.0 and len(_rdl_val_pos_df) > 0 and len(_rdl_val_neg_df) > 0:

    _n_pos_target_val = int(round((_rdl_train_pos_rate / (1.0 - _rdl_train_pos_rate)) * len(_rdl_val_neg_df)))

    _n_pos_target_val = min(_n_pos_target_val, len(_rdl_val_pos_df))

    if _n_pos_target_val < len(_rdl_val_pos_df):

        _rng_val = np.random.default_rng(42)

        _keep_pos_idx_val = _rng_val.choice(_rdl_val_pos_df.index.to_numpy(), size=_n_pos_target_val, replace=False)

        _keep_idx_val = np.sort(np.concatenate([_rdl_val_neg_df.index.to_numpy(), _keep_pos_idx_val]))

        rdl_val_table.df = rdl_val_table.df.loc[_keep_idx_val].copy()



print("=" * 70)

print("AT-RISK USER FILTERING STRATEGY (TARGET: 1-2% ANOMALY RATE)")

print("=" * 70)

print("At-risk definition (users to DROP):")

print(f"  - Recent txns <= {rdl_at_risk_thresholds['recent_max']} (bottom 10% of non-churners)")

print(f"  - OR days_since_last_txn >= {rdl_at_risk_thresholds['min_days_since']} (top 90% of non-churners)")

print()

print(f"Train: kept {int(rdl_train_keep.sum()):,} users (dropped {(~rdl_train_keep).sum():,} at-risk)")

print(f"  -> Pre-downsample: {int(rdl_train_anomaly_labels[rdl_train_keep].sum()):,} anomalies")

print(f"  -> Post-downsample: {int(rdl_train_table.df['churn'].sum()):,} anomalies ({rdl_train_table.df['churn'].mean():.4f})")

print()

print(f"Val:   kept {int(rdl_val_keep.sum()):,} users (dropped {(~rdl_val_keep).sum():,} at-risk)")

print(f"  -> Pre-downsample: {int(rdl_val_anomaly_labels[rdl_val_keep].sum()):,} anomalies")

print(f"  -> Post-downsample: {int(rdl_val_table.df['churn'].sum()):,} anomalies ({rdl_val_table.df['churn'].mean():.4f})")

print("=" * 70)


AT-RISK USER FILTERING STRATEGY (TARGET: 1-2% ANOMALY RATE)
At-risk definition (users to DROP):
  - Recent txns <= 1 (bottom 10% of non-churners)
  - OR days_since_last_txn >= 12 (top 90% of non-churners)

Train: kept 796,646 users (dropped 136,104 at-risk)
  -> Pre-downsample: 238,007 anomalies
  -> Post-downsample: 8,507 anomalies (0.0150)

Val:   kept 15,503 users (dropped 2,872 at-risk)
  -> Pre-downsample: 4,044 anomalies
  -> Post-downsample: 174 anomalies (0.0150)


In [19]:
# ── Random Forest rule mining for at-risk-filtered anomalies ───────────────────
import json, re
from sklearn.ensemble import RandomForestClassifier

# Build RF table directly from current filtered train table.
_activity_train_for_rf = _activity_features_for_targets(rdl_train_table.df, lookback_days=30)
_flat = _build_customer_flat(rdl_train_table.df, _activity_train_for_rf)
_flat["churn"] = rdl_train_table.df["churn"].to_numpy(dtype=np.int64)

print(f"RF table shape: {_flat.shape}")
print(f"RF anomaly rate: {_flat['churn'].mean():.4f}")

RF_PARAMS = dict(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
RF_MAX_TRAIN = 200_000
DROP_COLS = {"customer_id", "timestamp", "churn"}

feature_names = [c for c in _flat.columns if c not in DROP_COLS]
X_df = _flat[feature_names].copy()
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)
X = X_df.to_numpy(dtype=np.float32)
y_rf = _flat["churn"].to_numpy(dtype=np.int64)

rng = np.random.default_rng(42)
if len(X) > RF_MAX_TRAIN:
    pos_idx = np.where(y_rf > 0)[0]
    neg_idx = np.where(y_rf == 0)[0]
    pos_rate = float(y_rf.mean())
    pos_take = min(pos_idx.size, max(1, int(RF_MAX_TRAIN * pos_rate)))
    neg_take = min(neg_idx.size, RF_MAX_TRAIN - pos_take)
    sampled = rng.permutation(np.concatenate([
        pos_idx[rng.permutation(pos_idx.size)[:pos_take]],
        neg_idx[rng.permutation(neg_idx.size)[:neg_take]],
    ]))
    X_fit, y_fit = X[sampled], y_rf[sampled]
else:
    X_fit, y_fit = X, y_rf

print(f"RF training: {len(X_fit):,} rows  pos={int(y_fit.sum()):,}  features={X.shape[1]}")
clf = RandomForestClassifier(**RF_PARAMS)
clf.fit(X_fit, y_fit)
print("RF trained.")

# ── Extract root-to-leaf paths ────────────────────────────────────────────────
MIN_LEAF_SAMPLES = 5
MIN_POS_PROB = 0.50
POOL_TREES = 100


def _parent_map(tree_):
    parent = np.full(tree_.node_count, -1, np.int32)
    is_left = np.zeros(tree_.node_count, bool)
    for p in range(tree_.node_count):
        l, r = int(tree_.children_left[p]), int(tree_.children_right[p])
        if l >= 0:
            parent[l] = p
            is_left[l] = True
        if r >= 0:
            parent[r] = p
    return parent, is_left


def _leaf_sql(tree_est, leaf_id, fnames):
    tree_ = tree_est.tree_
    parent, is_left = _parent_map(tree_)
    conds, node = [], int(leaf_id)
    while parent[node] != -1:
        p = int(parent[node])
        fidx = int(tree_.feature[p])
        if fidx >= 0:
            op = "<=" if is_left[node] else ">"
            conds.append(f"{fnames[fidx]} {op} {tree_.threshold[p]:.6g}")
        node = p
    conds.reverse()
    return " AND ".join(conds)


all_rows = []
leaf_assignments = np.stack([est.apply(X_fit) for est in clf.estimators_[:POOL_TREES]], axis=1)

for tid, tree_est in enumerate(clf.estimators_[:POOL_TREES]):
    tree_ = tree_est.tree_
    row_leaf = leaf_assignments[:, tid]
    for leaf_id in np.where(tree_.children_left == -1)[0]:
        if int(tree_.n_node_samples[leaf_id]) < MIN_LEAF_SAMPLES:
            continue
        vals = tree_.value[leaf_id][0]
        pos_prob = float(vals[1] / vals.sum()) if (vals.sum() > 0 and vals.shape[0] > 1) else 0.0
        if pos_prob < MIN_POS_PROB:
            continue
        sql = _leaf_sql(tree_est, leaf_id, feature_names)
        if not sql:
            continue
        mask = row_leaf == leaf_id
        all_rows.append({
            "rule_id": f"rule_{len(all_rows)+1:04d}",
            "tree_id": tid,
            "pos_prob": pos_prob,
            "val_support": int(mask.sum()),
            "val_precision": float(y_fit[mask].mean()) if mask.any() else 0.0,
            "sql_predicate": sql,
        })

rule_paths_df = (
    pd.DataFrame(all_rows)
    .sort_values(["val_precision", "val_support"], ascending=False)
    .drop_duplicates(subset=["sql_predicate"])
    .reset_index(drop=True)
)
print(f"Extracted {len(rule_paths_df):,} unique rule candidates.")
print(rule_paths_df.head(5)[["rule_id", "val_precision", "val_support", "sql_predicate"]])


def _sample_rules(df, n=30, seed=42):
    pool = df[df["val_support"] >= 1].head(2000).copy()
    if pool.empty:
        return []
    w = (np.sqrt(pool["val_precision"].clip(0)) * np.log1p(pool["val_support"].clip(0))).to_numpy() + 1e-9
    w /= w.sum()
    pick = np.random.default_rng(seed).choice(len(pool), size=min(n, len(pool)), replace=False, p=w)
    return pool.iloc[sorted(pick)].to_dict(orient="records")


seed_rules = _sample_rules(rule_paths_df, n=30)
print(f"Sampled {len(seed_rules)} seed rules for LLM.")

RF table shape: (567146, 14)
RF anomaly rate: 0.0150
RF training: 200,000 rows  pos=2,999  features=11
RF trained.
Extracted 3,805 unique rule candidates.
     rule_id  val_precision  val_support  \
0  rule_0001            1.0          115   
1  rule_0300            1.0          115   
2  rule_0766            1.0          115   
3  rule_1147            1.0          115   
4  rule_1335            1.0          115   

                                       sql_predicate  
0                               txn_span_days <= 1.5  
1  recent_txns_30d > 9.5 AND txn_span_days <= 337...  
2  recent_txns_30d > 9.5 AND days_since_last_txn ...  
3  days_since_last_txn <= 3.5 AND txn_span_days <...  
4  days_since_last_txn <= 3.5 AND recent_txns_90d...  
Sampled 30 seed rules for LLM.


In [20]:
# ── LLM rule generation (Llama 3.1-8B-Instruct, 4-bit) ───────────────────────
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
total_gib = int(torch.cuda.get_device_properties(0).total_memory / (1024**3))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_enable_fp32_cpu_offload=True,
    ),
    device_map="auto",
    max_memory={0: f"{max(4, int(total_gib*0.9))}GiB", "cpu": "64GiB"},
    low_cpu_mem_usage=True,
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print(f"Llama loaded (GPU budget={int(total_gib*0.9)}GiB).")

SCHEMA = """
Table: features_table  (customer-level activity for H&M user-churn detection)
Columns:
  recent_txns_30d      INTEGER  -- reviews in last 30 days
  days_since_last_txn  INTEGER  -- days since most recent review (9999=never)
  total_txns           INTEGER  -- cumulative reviews before churn timestamp
  recent_txns_90d      INTEGER  -- reviews in last 90 days
  mean_price             FLOAT    -- average star rating (1-5)
  price_std              FLOAT    -- std-dev of star ratings
  min_price              FLOAT    -- minimum star rating given
  max_price              FLOAT    -- maximum star rating given
  distinct_articles       INTEGER  -- distinct products reviewed
  txn_span_days        INTEGER  -- days between first and last review
  low_price_flag         INTEGER  -- 1 if mean_price < 3.0
""".strip()

# Seed-rule fallback: continue even if RF mining cell did not define seed_rules.
if "seed_rules" not in globals():
    if "rule_paths_df" in globals() and hasattr(rule_paths_df, "empty") and not rule_paths_df.empty:
        seed_rules = rule_paths_df.head(30).to_dict(orient="records")
    else:
        seed_rules = []
        print("Warning: seed_rules unavailable; prompting LLM without RF seeds.")

seed_text = json.dumps([{
    "rule_id": r.get("rule_id", f"seed_{i:03d}"),
    "sql_predicate": r.get("sql_predicate", ""),
    "val_precision": round(float(r.get("val_precision", 0)), 4),
    "val_support": int(r.get("val_support", 0)),
} for i, r in enumerate(seed_rules[:20])], indent=2)

outputs = pipe(
    [
        {"role": "system", "content": (
            "You write SQL WHERE-clause predicates for an H&M user anomaly detector. "
            "The positives are churn anomalies after at-risk users are removed. "
            "You receive RF seed rules and must generate refined or novel predicates. "
            "Return only a valid JSON array. No prose."
        )},
        {"role": "user", "content": (
            f"SCHEMA:\n{SCHEMA}\n\n"
            f"SEED RULES:\n{seed_text}\n\n"
            "Generate 15 SQL WHERE predicates. Each object must have keys: "
            "\"candidate_id\" (string), \"sql_predicate\" (SQL), "
            "\"mutation_type\" (tighten|relax|combine|novel), \"rationale\" (one sentence). "
            "Return only the JSON array."
        )},
    ],
    max_new_tokens=2500, do_sample=False, temperature=0.0, top_p=1.0,
)
response_text = outputs[0]["generated_text"][-1]["content"]
print("LLM response (first 500 chars):")
print(response_text[:500])


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

<REPO_ROOT>/rad/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Llama loaded (GPU budget=42GiB).


<REPO_ROOT>/rad/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


LLM response (first 500 chars):
[
  {
    "candidate_id": "pred_1",
    "sql_predicate": "distinct_articles > 8 AND min_price > 0.00622881 AND distinct_articles <= 28 AND recent_txns_30d > 9.5 AND distinct_articles <= 18 AND txn_span_days <= 2.5 AND txn_span_days <= 1.5",
    "mutation_type": "tighten",
    "rationale": "Tightened the condition on distinct_articles to improve precision."
  },
  {
    "candidate_id": "pred_2",
    "sql_predicate": "recent_txns_90d <= 6 AND txn_span_days <= 1",
    "mutation_type": "tighten",
  


In [21]:
# ── Parse LLM output + evaluate SQL predicates via DuckDB ────────────────────
import ast, re, json, duckdb

SQL_KW = {"AND","OR","NOT","IN","IS","NULL","TRUE","FALSE","BETWEEN","LIKE"}

def _parse_llm(text):
    s = text.strip()
    # Strip markdown code fences.
    m = re.match(r"^```(?:json)?\s*(.*?)\s*```$", s, re.DOTALL | re.IGNORECASE)
    if m: s = m.group(1).strip()
    s = s.replace("\u201c",'"').replace("\u201d",'"').replace("\u2018","'").replace("\u2019","'")
    s = re.sub(r",\s*([}\]])", r"\1", s)
    s = re.sub(r"\bTrue\b", "true", s); s = re.sub(r"\bFalse\b", "false", s)
    # Try parsing the whole string directly first.
    try:
        result = json.loads(s)
        if isinstance(result, list): return result
    except Exception:
        pass
    # Find first [...] using depth tracking.
    start = s.find("[")
    if start == -1: return []
    depth, in_str, esc = 0, False, False
    for i in range(start, len(s)):
        c = s[i]
        if in_str:
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                in_str = False
        elif c == '"':
            in_str = True
        elif c == "[":
            depth += 1
        elif c == "]":
            depth -= 1
            if depth == 0:
                chunk = s[start:i+1]
                try: return json.loads(chunk)
                except Exception:
                    try: return ast.literal_eval(chunk)
                    except Exception: return []
    return []

llm_candidates = _parse_llm(response_text)
print(f"Parsed {len(llm_candidates)} LLM candidates.")

if not llm_candidates:
    print("LLM parsing failed — using seed rules as fallback.")
    llm_candidates = [
        {"candidate_id": f"seed_{i:03d}", "sql_predicate": r["sql_predicate"],
         "mutation_type": "seed_fallback", "rationale": "RF seed rule"}
        for i, r in enumerate(seed_rules[:20]) if r.get("sql_predicate","").strip()
    ]

# Flat feature table for SQL eval from CURRENT filtered train table.
_activity_train_for_eval = _activity_features_for_targets(rdl_train_table.df, lookback_days=30)
eval_df = _build_customer_flat(rdl_train_table.df, _activity_train_for_eval)
eval_df = eval_df.drop(columns=["timestamp"], errors="ignore").copy()
eval_df["customer_id"] = eval_df["customer_id"].astype(np.int64)

# Pad any missing columns referenced in predicates.
ref_cols = set()
for cand in llm_candidates:
    for tok in re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", str(cand.get("sql_predicate",""))):
        if tok.upper() not in SQL_KW: ref_cols.add(tok)
for c in sorted(ref_cols - set(eval_df.columns) - {"customer_id"}):
    eval_df[c] = 0

all_cids  = eval_df["customer_id"].to_numpy(dtype=np.int64)
result_df = pd.DataFrame({"customer_id": all_cids})

con = duckdb.connect(":memory:")
con.register("features_table", eval_df)

parsed_rules_rows, used_names, n_matched = [], {"customer_id"}, 0

for i, cand in enumerate(llm_candidates):
    raw  = cand.get("candidate_id", f"llm_{i:03d}")
    col  = re.sub(r"[^0-9A-Za-z_]+","_", str(raw)).strip("_") or f"rule_{i:03d}"
    base, sfx = col, 1
    while col in used_names: col = f"{base}_{sfx}"; sfx += 1
    used_names.add(col)

    result_df[col] = np.int8(0)
    pred = cand.get("sql_predicate", None)
    if isinstance(pred, str) and pred.strip():
        try:
            matched_ids = (
                pd.to_numeric(
                    con.execute(f"SELECT customer_id FROM features_table WHERE ({pred})").fetchdf()["customer_id"],
                    errors="coerce"
                ).dropna().astype(np.int64).unique()
            )
            mask = result_df["customer_id"].isin(matched_ids)
            result_df.loc[mask, col] = np.int8(1)
            n_matched += 1
            print(f"  {raw}: {int(mask.sum())} matches")
        except Exception as e:
            print(f"  {raw} SQL error: {str(e)[:160]}")
    parsed_rules_rows.append({"candidate_id": raw, "col_name": col,
                               "sql_predicate": pred, "mutation_type": cand.get("mutation_type","")})

con.close()
print(f"\nRules with matches: {n_matched}/{len(llm_candidates)}")

parsed_rules_df = pd.DataFrame(parsed_rules_rows)
os.makedirs("data", exist_ok=True)
parsed_rules_df.to_csv("data/hm_parsed_rules_df.csv", index=False)
rule_cols = [c for c in result_df.columns if c != "customer_id"]
print(f"Rule columns: {rule_cols}")

Parsed 15 LLM candidates.
  pred_1: 163 matches
  pred_2: 77 matches
  pred_3: 24 matches
  pred_4: 22 matches
  pred_5: 32 matches
  pred_6: 36 matches
  pred_7: 503 matches
  pred_8: 187 matches
  pred_9: 382 matches
  pred_10: 6126 matches
  pred_11: 1888 matches
  pred_12: 47 matches
  pred_13: 875 matches
  pred_14: 231 matches
  pred_15: 9872 matches

Rules with matches: 15/15
Rule columns: ['pred_1', 'pred_2', 'pred_3', 'pred_4', 'pred_5', 'pred_6', 'pred_7', 'pred_8', 'pred_9', 'pred_10', 'pred_11', 'pred_12', 'pred_13', 'pred_14', 'pred_15']


In [22]:
# ── Align rule violation matrix to customer node indices ──────────────────────
# Mirrors RAD_fixed cell 16: auth_id → node index alignment.

RULE_TABLE_CACHE = os.path.join("data", "hm_customer_rule_table.pt")

customer_df  = rdl_db.table_dict["customer"].df.copy()
customer_df["customer_id"] = customer_df["customer_id"].astype(np.int64)
cid_to_nidx  = {int(cid): i for i, cid in enumerate(customer_df["customer_id"].tolist())}
n_cust_nodes = len(cid_to_nidx)
n_rules      = len(rule_cols)

print(f"Customer nodes: {n_cust_nodes:,}  |  rule features: {n_rules}")

rule_table = torch.zeros(n_cust_nodes, max(n_rules, 1), dtype=torch.float32)
if n_rules > 0:
    mat  = result_df[rule_cols].to_numpy(dtype=np.float32)
    cids = result_df["customer_id"].to_numpy(dtype=np.int64)
    for row_i, cid in enumerate(cids):
        nidx = cid_to_nidx.get(int(cid))
        if nidx is not None:
            rule_table[nidx] = torch.from_numpy(mat[row_i])
    coverage = int((rule_table[:, :n_rules].any(dim=1)).sum()) / n_cust_nodes
    print(f"Rule coverage: {coverage:.2%} of customer nodes have ≥1 match")

torch.save({"rule_table": rule_table, "rule_cols": rule_cols, "n_rules": n_rules}, RULE_TABLE_CACHE)
print(f"Rule table saved → {RULE_TABLE_CACHE}  shape={tuple(rule_table.shape)}")

Customer nodes: 1,371,980  |  rule features: 15
Rule coverage: 0.61% of customer nodes have ≥1 match
Rule table saved → data/hm_customer_rule_table.pt  shape=(1371980, 15)


## RAD Model with Rule Augmentation

Rule-violation indicators are projected into the customer latent space as an additive bias
after the GNN — the mini-batch equivalent of RAD_fixed's pre-encoding feature concatenation.

In [23]:
# ── RADAutoencoder ────────────────────────────────────────────────────────────

class RADAutoencoder(torch.nn.Module):
    def __init__(self, data, col_stats_dict, num_layers, channels, aggr, norm,
                 rule_dim=0, latent_dim=48, decoder_num_layers=None, decoder_head_layers=2):
        super().__init__()
        if decoder_num_layers is None:
            decoder_num_layers = num_layers
        if latent_dim >= channels:
            raise ValueError(f"latent_dim ({latent_dim}) must be < channels ({channels}) for bottlenecking")

        self.channels = channels
        self.latent_dim = latent_dim
        self.rule_dim = rule_dim
        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={nt: data[nt].tf.col_names_dict for nt in data.node_types},
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[nt for nt in data.node_types if "time" in data[nt]], channels=channels)

        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.to_latent = nn.Linear(channels, latent_dim)
        self.decoder_gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=latent_dim,
            aggr=aggr,
            num_layers=decoder_num_layers,
        )
        self.from_latent = nn.Linear(latent_dim, channels)
        self.feat_decoder = MLP(
            channels,
            hidden_channels=channels,
            out_channels=channels,
            norm=norm,
            num_layers=decoder_head_layers,
        )
        self.rel_diag = nn.ParameterDict(
            {self._ek(et): nn.Parameter(torch.ones(latent_dim)) for et in data.edge_types})
        self.rule_proj = nn.Linear(rule_dim, latent_dim, bias=False) if rule_dim > 0 else None
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.decoder_gnn.reset_parameters()
        self.feat_decoder.reset_parameters()
        torch.nn.init.xavier_uniform_(self.to_latent.weight)
        torch.nn.init.zeros_(self.to_latent.bias)
        torch.nn.init.xavier_uniform_(self.from_latent.weight)
        torch.nn.init.zeros_(self.from_latent.bias)
        for p in self.rel_diag.values():
            torch.nn.init.ones_(p)
        if self.rule_proj is not None:
            torch.nn.init.xavier_uniform_(self.rule_proj.weight)

    @staticmethod
    def _ek(et):
        return f"{et[0]}__{et[1]}__{et[2]}"

    def _encode(self, batch, entity_table):
        seed_time = batch[entity_table].seed_time
        x = self.encoder(batch.tf_dict)
        for nt, rt in self.temporal_encoder(seed_time, batch.time_dict, batch.batch_dict).items():
            x[nt] = x[nt] + rt
        pre = {k: v.detach() for k, v in x.items()}
        z = self.gnn(
            x,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )
        z_latent = {k: self.to_latent(v) for k, v in z.items()}
        return pre, z_latent, seed_time.size(0)

    def forward(self, batch, entity_table, rule_table=None):
        pre, z_latent, n = self._encode(batch, entity_table)

        z_dec_in = {k: v for k, v in z_latent.items()}
        if rule_table is not None and self.rule_proj is not None:
            node_ids = batch[entity_table].n_id
            z_dec_in[entity_table] = z_dec_in[entity_table] + self.rule_proj(
                rule_table[node_ids].to(z_dec_in[entity_table].device)
            )

        z_dec = self.decoder_gnn(
            z_dec_in,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )
        seed_dec = z_dec[entity_table][:n]
        seed_dec = self.from_latent(seed_dec)
        return self.feat_decoder(seed_dec), pre[entity_table][:n], z_dec, n

    def score_edges(self, z, et, ei):
        k = self._ek(et)
        if k not in self.rel_diag or ei.numel() == 0:
            return ei.new_zeros(ei.size(1), dtype=torch.float32)
        d = self.rel_diag[k]
        return (z[et[0]][ei[0]] * d * z[et[2]][ei[1]]).sum(dim=1)

    def structure_bce_loss(self, z, batch, neg_ratio=1.0):
        losses = []
        for et in batch.edge_types:
            ei = batch[et].edge_index
            if ei.numel() == 0 or self._ek(et) not in self.rel_diag:
                continue
            ns, nd = int(z[et[0]].size(0)), int(z[et[2]].size(0))
            if ns == 0 or nd == 0:
                continue
            n_neg = max(1, int(neg_ratio * ei.size(1)))
            neg_ei = torch.stack([
                torch.randint(0, ns, (n_neg,), device=ei.device),
                torch.randint(0, nd, (n_neg,), device=ei.device),
            ])
            losses.append(0.5 * (
                F.binary_cross_entropy_with_logits(
                    self.score_edges(z, et, ei),
                    torch.ones_like(self.score_edges(z, et, ei)),
                )
                + F.binary_cross_entropy_with_logits(
                    self.score_edges(z, et, neg_ei),
                    torch.zeros_like(self.score_edges(z, et, neg_ei)),
                )
            ))
        return torch.stack(losses).mean() if losses else next(iter(z.values())).new_zeros(())

    def seed_struct_error(self, z, batch, n_seed, entity_table):
        dev = z[entity_table].device
        es = torch.zeros(n_seed, device=dev)
        ec = torch.zeros(n_seed, device=dev)
        for et in batch.edge_types:
            ei = batch[et].edge_index
            if ei.numel() == 0 or self._ek(et) not in self.rel_diag:
                continue
            err = F.softplus(-self.score_edges(z, et, ei))
            for side, idx_row in [(et[0], 0), (et[2], 1)]:
                if side == entity_table:
                    v = ei[idx_row] < n_seed
                    if v.any():
                        es.index_add_(0, ei[idx_row][v], err[v])
                        ec.index_add_(0, ei[idx_row][v], torch.ones(v.sum(), device=dev))
        return es / torch.clamp(ec, min=1.0)


def pairwise_rank_loss(scores, labels, max_pairs=32768):
    scores, labels = scores.view(-1), labels.view(-1)
    pos = scores[labels > 0.5]
    neg = scores[labels <= 0.5]
    if pos.numel() == 0 or neg.numel() == 0:
        return scores.new_zeros(()), 0
    n = min(int(pos.numel()) * int(neg.numel()), max_pairs)
    pi = torch.randint(0, int(pos.numel()), (n,), device=scores.device)
    ni = torch.randint(0, int(neg.numel()), (n,), device=scores.device)
    return F.softplus(-(pos[pi] - neg[ni])).mean(), n


print("RADAutoencoder defined.")

RADAutoencoder defined.


In [ ]:
# ── Hyperparameters + model build ─────────────────────────────────────────────
_hp = dict(
    channels=128,
    latent_dim=48,
    num_layers=2,
    decoder_num_layers=2,
    decoder_head_layers=2,
    aggr="sum",
    norm="batch_norm",
    lr=1e-3,
    weight_decay=1e-4,
    epochs=10,
    patience=3,
    max_steps_train=2000,
    grad_clip=2.0,
    lambda_struct=0.5,
    lambda_rank=1.0,
    rank_max_pairs=32768,
    neg_ratio=1.0,
    alpha_feat=1.0,
    alpha_struct=1.0,
)

RULE_TABLE_CACHE = os.path.join("data", "hm_customer_rule_table.pt")
_disable_rules = os.environ.get("RAD_DISABLE_RULES", "0").strip() == "1"
if _disable_rules:
    rule_table, rule_cols, n_rules = None, [], 0
    print("RAD rules disabled via RAD_DISABLE_RULES=1.")
elif os.path.exists(RULE_TABLE_CACHE):
    _rt = torch.load(RULE_TABLE_CACHE, weights_only=False)
    rule_table = _rt["rule_table"]
    rule_cols = _rt["rule_cols"]
    n_rules = _rt["n_rules"]
    print(f"Rule table loaded: {tuple(rule_table.shape)}")
else:
    rule_table, rule_cols, n_rules = None, [], 0
    print("No rule table — running without rule augmentation.")

seed_everything(42)
rad_model = RADAutoencoder(
    rdl_data,
    rdl_col_stats_dict,
    num_layers=_hp["num_layers"],
    channels=_hp["channels"],
    aggr=_hp["aggr"],
    norm=_hp["norm"],
    rule_dim=n_rules,
    latent_dim=_hp["latent_dim"],
    decoder_num_layers=_hp["decoder_num_layers"],
    decoder_head_layers=_hp["decoder_head_layers"],
).to(device)
rad_opt = torch.optim.AdamW(rad_model.parameters(), lr=_hp["lr"], weight_decay=_hp["weight_decay"])
print(
    f"RADAutoencoder | channels={_hp['channels']} | latent_dim={_hp['latent_dim']} | "
    f"enc_layers={_hp['num_layers']} | dec_layers={_hp['decoder_num_layers']} | "
    f"dec_head={_hp['decoder_head_layers']} | rule_dim={n_rules} | "
    f"params={sum(p.numel() for p in rad_model.parameters() if p.requires_grad):,}"
)"}}]}

Rule table loaded: (1371980, 15)
RADAutoencoder | channels=128 | latent_dim=48 | enc_layers=2 | dec_layers=2 | dec_head=2 | rule_dim=15 | params=2,295,552


In [25]:
# ── Training + evaluation ─────────────────────────────────────────────────────

_val_labels    = rdl_val_table.df["churn"].astype(np.int64).to_numpy()
_rt_gpu        = rule_table.to(device) if rule_table is not None else None

# Rebuild loaders in-process to avoid notebook multiprocessing worker crashes.
from torch_geometric.loader import NeighborLoader
_safe_loader_dict = {}
for _split, _table in [("train", rdl_train_table), ("val", rdl_val_table), ("test", rdl_test_table)]:
    _ti = get_node_train_table_input(table=_table, task=rdl_task)
    _safe_loader_dict[_split] = NeighborLoader(
        rdl_data,
        num_neighbors=[32, 32],
        time_attr="time",
        input_nodes=_ti.nodes,
        input_time=_ti.time,
        transform=_ti.transform,
        batch_size=1024,
        temporal_strategy="uniform",
        shuffle=(_split == "train"),
        num_workers=0,
        persistent_workers=False,
        pin_memory=True,
    )
rdl_loader_dict = _safe_loader_dict

def _train_epoch():
    rad_model.train()
    S = dict(loss=0., feat=0., struct=0., rank=0., n=0, pairs=0, steps=0)
    feat_buf, struct_buf = [], []  # buffer normal-seed errors → normalization stats

    for batch in tqdm(rdl_loader_dict["train"], desc="train", leave=False):
        batch = batch.to(device)
        rad_opt.zero_grad(set_to_none=True)

        recon, target, z, n = rad_model(batch, rdl_entity_table, _rt_gpu)
        lbl  = batch[rdl_entity_table].y.float().view(-1)
        norm = lbl < 0.5
        if not norm.any(): continue

        feat_loss   = F.mse_loss(recon[norm], target[norm])
        struct_loss = rad_model.structure_bce_loss(z, batch, _hp["neg_ratio"])
        feat_err    = ((recon - target) ** 2).mean(dim=1)
        struct_err  = rad_model.seed_struct_error(z, batch, n, rdl_entity_table)
        combined    = _hp["alpha_feat"] * feat_err + _hp["alpha_struct"] * struct_err
        rank_loss, np_ = pairwise_rank_loss(combined, lbl, _hp["rank_max_pairs"])

        loss = feat_loss + _hp["lambda_struct"] * struct_loss + _hp["lambda_rank"] * rank_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rad_model.parameters(), _hp["grad_clip"])
        rad_opt.step()

        cnt = int(norm.sum())
        S["loss"]   += float(loss.detach())        * cnt
        S["feat"]   += float(feat_loss.detach())   * cnt
        S["struct"] += float(struct_loss.detach()) * cnt
        S["rank"]   += float(rank_loss.detach())   * cnt
        S["n"] += cnt; S["pairs"] += np_; S["steps"] += 1

        # Collect normal-seed errors for normalization stats — free, no extra pass.
        feat_buf.append(feat_err[norm].detach().cpu())
        struct_buf.append(struct_err[norm].detach().cpu())

        if _hp["max_steps_train"] and S["steps"] >= _hp["max_steps_train"]: break

    if S["n"] == 0: raise RuntimeError("No normal training samples.")

    all_f = torch.cat(feat_buf).numpy(); all_s = torch.cat(struct_buf).numpy()
    norm_stats = dict(feat_mu=float(np.mean(all_f)),   feat_sd=float(np.std(all_f)),
                      struct_mu=float(np.mean(all_s)), struct_sd=float(np.std(all_s)))
    return {k: (v/S["n"] if k in ("loss","feat","struct","rank") else v) for k,v in S.items()}, norm_stats


@torch.no_grad()
def _score(loader, max_batches=None):
    rad_model.eval()
    feats, structs = [], []
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches: break
        batch = batch.to(device)
        recon, target, z, n = rad_model(batch, rdl_entity_table, _rt_gpu)
        feats.append(((recon - target)**2).mean(dim=1).cpu())
        structs.append(rad_model.seed_struct_error(z, batch, n, rdl_entity_table).cpu())
    return torch.cat(feats).numpy(), torch.cat(structs).numpy()


def _combine(fe, se, ns):
    fz = (fe - ns["feat_mu"])   / max(ns["feat_sd"],   1e-8)
    sz = (se - ns["struct_mu"]) / max(ns["struct_sd"], 1e-8)
    return _hp["alpha_feat"] * fz + _hp["alpha_struct"] * sz


# ── Train ─────────────────────────────────────────────────────────────────────
best_metric, best_state, bad = -math.inf, None, 0
norm_stats = dict(feat_mu=0., feat_sd=1., struct_mu=0., struct_sd=1.)

_log = "/tmp/rad_hm_user_churn_log.txt"
with open(_log, "w") as f: f.write(f"RAD H&M user-churn | rules={n_rules}\n")

for epoch in range(1, _hp["epochs"] + 1):
    tr, norm_stats = _train_epoch()  # norm_stats from in-loop buffers — no extra pass

    vf, vs   = _score(rdl_loader_dict["val"])  # 384 batches — fast
    val_pred = _combine(vf, vs, norm_stats)

    val_met  = rdl_task.evaluate(val_pred, rdl_val_table)
    val_aupr = average_precision_score(_val_labels, val_pred)

    line = (f"Epoch {epoch:02d}/{_hp['epochs']} | steps={tr['steps']} | "
            f"loss={tr['loss']:.4f} | feat={tr['feat']:.4f} | struct={tr['struct']:.4f} | "
            f"rank={tr['rank']:.4f} | pairs={tr['pairs']} | "
            f"val_auprc={val_aupr:.4f} | val {val_met}")
    print(line)
    with open(_log, "a") as f: f.write(line + "\n")

    if val_met[rdl_tune_metric] > best_metric:
        best_metric = val_met[rdl_tune_metric]
        best_state  = {k: v.detach().cpu().clone() for k,v in rad_model.state_dict().items()}
        best_norm   = dict(norm_stats)
        bad = 0
    else:
        bad += 1
        if bad >= _hp["patience"]:
            print(f"Early stopping at epoch {epoch}.")
            break

rad_model.load_state_dict(best_state)
print(f"\nBest val {rdl_tune_metric}: {best_metric:.4f}")

ValueError: Input contains NaN.

In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────────

ns = best_norm  # normalization stats from best checkpoint's epoch

# Validation on at-risk-filtered anomaly labels used for training.
vf, vs   = _score(rdl_loader_dict["val"])
val_pred = _combine(vf, vs, ns)
val_met  = rdl_task.evaluate(val_pred, rdl_val_table)
val_aupr = average_precision_score(_val_labels, val_pred)
val_auroc = roc_auc_score(_val_labels, val_pred) if np.unique(_val_labels).size > 1 else float("nan")

# Evaluation on original (unfiltered) validation labels.
_orig_lbl = _rdl_orig_val_table.df["churn"].astype(np.int64).to_numpy()
orig_input = get_node_train_table_input(table=_rdl_orig_val_table, task=rdl_task)
orig_val_loader = NeighborLoader(
    rdl_data,
    num_neighbors=[32, 32],
    time_attr="time",
    input_nodes=orig_input.nodes,
    input_time=orig_input.time,
    transform=orig_input.transform,
    batch_size=1024,
    temporal_strategy="uniform",
    shuffle=False,
    prefetch_factor=1,
    pin_memory=True,
    num_workers=2,
    persistent_workers=False,
)
orig_vf, orig_vs = _score(orig_val_loader)
orig_val_pred = _combine(orig_vf, orig_vs, ns)
orig_met  = rdl_task.evaluate(orig_val_pred, _rdl_orig_val_table)
orig_aupr = average_precision_score(_orig_lbl, orig_val_pred)
orig_auroc = roc_auc_score(_orig_lbl, orig_val_pred) if np.unique(_orig_lbl).size > 1 else float("nan")

tf, ts    = _score(rdl_loader_dict["test"])
test_pred = _combine(tf, ts, ns)

print("\n" + "=" * 70)
print("RAD H&M User-Churn + Rule Augmentation — Final Results")
print("=" * 70)
print(f"Rule features:                 {n_rules}")
print(f"At-risk thresholds:            {rdl_at_risk_thresholds}")
print(f"Val (at-risk filtered labels): {val_met}  auprc={val_aupr:.4f}  auroc={val_auroc:.4f}")
print(f"Val (original labels):         {orig_met}  auprc={orig_aupr:.4f}  auroc={orig_auroc:.4f}")
print(f"Test scores:                   mean={np.mean(test_pred):.4f}  std={np.std(test_pred):.4f}")
print("=" * 70)

os.makedirs("weights", exist_ok=True)
torch.save({
    "model_state_dict": rad_model.state_dict(),
    "optimizer_state_dict": rad_opt.state_dict(),
    "best_val_metric": best_metric,
    "norm_stats": ns,
    "hyperparams": _hp,
    "at_risk_thresholds": rdl_at_risk_thresholds,
    "rule_cols": rule_cols,
    "n_rules": n_rules,
}, "weights/rad_hm_user_churn.pt")
print("Saved → weights/rad_hm_user_churn.pt")
